# Segmentación de clientes mayoristas
Grupo 9 · Evaluación 3

Fuente: Margarida Cardoso (2013), [Wholesale customers](https://doi.org/10.24432/C5030X), UCI, CC BY 4.0. Este notebook usa el mismo módulo que la ejecución por consola. Abrir desde la carpeta Implementacion.

## 1. Validación de la fuente
El identificador de fila sirve solo dentro de esta versión del archivo. Los atípicos por IQR se marcan y se conservan.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import pandas as pd
from IPython.display import display
from segmentacion import SOURCE, preparar, ejecutar

base, calidad = preparar(SOURCE)
print(json.dumps(calidad, indent=2))
display(base.head(3))

{
  "filas": 440,
  "columnas": 8,
  "nulos": 0,
  "duplicados": 0,
  "negativos": 0,
  "no_finitos": 0,
  "codigos_invalidos": 0,
  "atipicos": 108,
  "sha256": "c3d018c643565b85cee733c4a2ac76dd76e080e857cb23f0ccfcc2e15a6c17ef",
  "version": "1.0"
}


,cliente_id,canal,region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,atipico_iqr
0,CL-0001,Retail,Otra,12669,9656,7561,214,2674,1338,False
1,CL-0002,Retail,Otra,7057,9810,9568,1762,3293,1776,False
2,CL-0003,Retail,Otra,6353,8808,7684,2405,3516,7844,True


## 2. Entrenamiento y verificación
Se entrenan cinco soluciones sobre las seis variables de gasto con log1p, estandarización y K-means. La ejecución del notebook usa una carpeta temporal para preservar las evidencias de la entrega.

In [2]:
sesion = TemporaryDirectory(prefix="mayoristas-")
salida = Path(sesion.name)
ejecucion = ejecutar(out=salida)
print('k seleccionado:', ejecucion['k'])
print('Modelo guardado y recargado:', ejecucion['prediccion_verificada'])
display(pd.read_csv(salida / 'comparacion_k.csv').round(6))
display(pd.read_csv(salida / 'estabilidad.csv').round(6))

k seleccionado: 2
Modelo guardado y recargado: True


,k,silhouette
0,2,0.290328
1,3,0.259416
2,4,0.188495
3,5,0.191585
4,6,0.200887


,semilla,ari
0,0,1.0
1,7,1.0
2,24,1.0
3,99,1.0


## 3. Perfiles y auditoría
Canal y región se usan para evaluar composición. Los cruces no miden efectos sobre el servicio.

In [3]:
clientes = pd.read_csv(salida / 'clientes_segmentados.csv')
assert len(clientes) == clientes.cliente_id.nunique() == len(base)
assert clientes.segmento.notna().all()
assert clientes.atipico_iqr.sum() == calidad['atipicos']
display(pd.read_csv(salida / 'perfiles.csv'))
display(pd.read_csv(salida / 'auditoria_canal.csv'))
display(pd.read_csv(salida / 'auditoria_region.csv'))
display(clientes.loc[clientes.cliente_id.eq('CL-0001')])
print('Asignación completa, identificadores únicos y atípicos conservados.')
sesion.cleanup()

,segmento,perfil,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,clientes
0,1,Frescos y congelados,10347.5,1871.0,2406.0,2225.0,327.5,749.5,252
1,2,"Lácteos, abarrotes e higiene",5406.5,7690.5,11527.0,1061.5,4495.0,1388.5,188


,canal,1,2
0,Horeca,244,54
1,Retail,8,134


,region,1,2
0,Lisboa,48,29
1,Oporto,28,19
2,Otra,176,140


,cliente_id,canal,region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,atipico_iqr,segmento,perfil
0,CL-0001,Retail,Otra,12669,9656,7561,214,2674,1338,False,2,"Lácteos, abarrotes e higiene"


Asignación completa, identificadores únicos y atípicos conservados.


## 4. Interpretación
Los perfiles describen compras anuales. La separación limitada y la composición desigual requieren revisión con usuarios antes de recomendar cambios de servicio. Para frecuencia o deriva temporal se requieren facturas fechadas.